# Comparación de inconsistencias — Docentes y Estudiantes (2025 vs 2026)

Notebook **único** de inconsistencias 2025 vs 2026 (docentes + estudiantes).

Objetivo: ver, por tipo, **qué es inconsistente entre 2025 y 2026**:

- % de **valores/categorías** que aparecen en un solo año (por variable)
- % de **filas** involucradas por esos valores
- qué **grupos de valores** están solo en una tabla
- qué **combinaciones** difieren en cantidades numéricas

Usa funciones de `compare_datasets_generic.py` (incl. `pct_desfasajes_categorias`, `pct_combinaciones_cantidad`, `tops_desfasajes`).

Prerrequisito: CSV limpios de `estructura.py` en `processed/`.


In [ ]:
import pathsetup  # raíz del repo en sys.path (notebooks en reportes/)
from pathlib import Path
from IPython.display import display
import pandas as pd

from compare_datasets_generic import (
    cargar,
    preparar_para_comparar,
    compare_datasets,
    reporte_comparacion,
    detectar_desfasajes,
    detectar_duplicados,
    reporte_duplicados,
    limpiar_duplicados_df,
    pct_desfasajes_categorias,
    pct_combinaciones_cantidad,
    tops_desfasajes,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 40)


In [ ]:
# ---- Config ----
USAR_LIMPIOS = True  # True: CSV limpios ; False: .xlsx crudos
GUARDAR_DEDUP = False  # True: escribe *_dedup.csv (como en notebooks viejos)

CARPETA_DATOS = Path.home() / "Downloads" / "Datos Ceibal 2025-2026 ver final"
assert CARPETA_DATOS.exists(), f"No existe: {CARPETA_DATOS}"
SALIDA = CARPETA_DATOS / "processed"

TIPOS = {
    "docentes": {
        "clean": ("docentes_2025_clean.csv", "docentes_2026_clean.csv"),
        "raw": ("datosUCU2025_doc.xlsx", "datosUCU2026_doc.xlsx"),
    },
    "estudiantes": {
        "clean": ("estudiantes_2025_clean.csv", "estudiantes_2026_clean.csv"),
        "raw": ("datosUCU2025_estu.xlsx", "datosUCU2026_estu.xlsx"),
    },
}

print("Carpeta:", CARPETA_DATOS)
print("USAR_LIMPIOS:", USAR_LIMPIOS)


## 0. Carga (ambos tipos)

Misma lógica que los notebooks de comparación, en un solo loop.


In [ ]:
datos = {}
for tipo, rutas in TIPOS.items():
    clave = "clean" if USAR_LIMPIOS else "raw"
    base = SALIDA if USAR_LIMPIOS else CARPETA_DATOS
    r25, r26 = (base / rutas[clave][0], base / rutas[clave][1])
    d25, d26 = cargar(r25), cargar(r26)
    datos[tipo] = {"df_25": d25, "df_26": d26, "ruta_25": r25, "ruta_26": r26}
    print(f"{tipo:12} 2025={d25.shape[0]:,}x{d25.shape[1]}  2026={d26.shape[0]:,}x{d26.shape[1]}")
    print(f"             <- {r25.name} | {r26.name}")


## 1. Preparación + reporte de comparación (texto)

`preparar_para_comparar` excluye columnas vacías en un año y homogeneiza dtypes mixtos.


In [ ]:
for tipo, pack in datos.items():
    d25, d26 = pack["df_25"], pack["df_26"]
    c25, c26 = preparar_para_comparar(d25, d26, verbose=True)
    pack["c25"], pack["c26"] = c25, c26
    print("\n" + "#" * 70)
    print(f"# {tipo.upper()}")
    print("#" * 70)
    print(reporte_comparacion(c25, c26, name1="2025", name2="2026"))


## 2. Detalle estructurado: categorías solo en un año + cantidades distintas


In [ ]:
for tipo, pack in datos.items():
    resultado = compare_datasets(pack["c25"], pack["c26"], name1="2025", name2="2026")
    pack["resultado"] = resultado
    print(f"\n=== {tipo.upper()} — resumen ===")
    display(pd.Series(resultado["resumen"]))
    print("Categorías desfasadas (valores solo en un año):")
    display(resultado["categorias_desfasadas"])
    print("Cantidades desfasadas (muestra):")
    display(resultado["cantidades_desfasadas"].head(20))


## 3. Duplicados por año (reporte + muestra)


In [ ]:
for tipo, pack in datos.items():
    d25, d26 = pack["df_25"], pack["df_26"]
    print(f"\n### {tipo}")
    print(reporte_duplicados(d25))
    print()
    print(reporte_duplicados(d26))
    mask = detectar_duplicados(d25, keep=False)
    print(f"Muestra duplicados 2025 ({int(mask.sum())} filas en máscara keep=False):")
    display(d25[mask].head(10))


## 4. Dedup en memoria → desfasajes operables

Igual que §4–5 de los notebooks: limpiar duplicados y recalcular inconsistencias con `n_filas`.


In [ ]:
for tipo, pack in datos.items():
    d25, d26 = pack["df_25"], pack["df_26"]
    print(f"\n### Dedup {tipo}")
    s25 = limpiar_duplicados_df(d25, verbose=True)
    s26 = limpiar_duplicados_df(d26, verbose=True)
    if GUARDAR_DEDUP:
        SALIDA.mkdir(parents=True, exist_ok=True)
        s25.to_csv(SALIDA / f"{tipo}_2025_dedup.csv", index=False)
        s26.to_csv(SALIDA / f"{tipo}_2026_dedup.csv", index=False)
        print("  guardados *_dedup.csv")

    c25, c26 = preparar_para_comparar(s25, s26, verbose=False)
    des = detectar_desfasajes(c25, c26, name1="2025", name2="2026", con_filas=True)
    pack.update({
        "sin_dups_25": s25, "sin_dups_26": s26,
        "c25_dedup": c25, "c26_dedup": c26,
        "desfasajes_dedup": des,
        "desfasajes_df": des["categorias"],
    })
    print("Categorías desfasadas (filas):", des["categorias"].shape[0])
    print("Cantidades desfasadas (filas):", des["cantidades"].shape[0])
    resumen_cat = (
        des["categorias"].groupby(["columna", "solo_en"]).size().unstack(fill_value=0)
        if len(des["categorias"]) else pd.DataFrame()
    )
    display(resumen_cat)


## 5. Porcentajes de inconsistencia (núcleo del reporte)

Por columna categórica desfasada:

| métrica | significado |
|---|---|
| `%_valores` | % de valores únicos de la columna que aparecen en **un solo** año |
| `%_filas` | % de filas (2025+2026 dedup) cuya categoría está solo en un año |

Además: % de **combinaciones** con al menos una cantidad numérica distinta entre años.


In [ ]:
impactos = []
for tipo, pack in datos.items():
    des_df = pack["desfasajes_df"]
    c25, c26 = pack["c25_dedup"], pack["c26_dedup"]
    pct_cat = pct_desfasajes_categorias(des_df, c25, c26)
    pct_cant = pct_combinaciones_cantidad(pack["desfasajes_dedup"])
    pack["pct_categorias"] = pct_cat
    pack["pct_cantidades"] = pct_cant

    print("\n" + "=" * 70)
    print(f" {tipo.upper()} — % por variable inconsistente")
    print("=" * 70)
    display(pct_cat)

    print(
        f"Combinaciones comunes: {pct_cant['combinaciones_comunes']:,} | "
        f"únicas con cantidad desfasada: {pct_cant['combinaciones_unicas_desfasadas']:,} "
        f"({pct_cant['%_combinaciones_desfasadas']}%)"
    )
    impactos.append({
        "tipo": tipo,
        "filas_2025": len(c25),
        "filas_2026": len(c26),
        "cols_con_categoria_desfasada": len(pct_cat),
        "%_filas_max_col": float(pct_cat["%_filas"].max()) if len(pct_cat) else 0.0,
        "col_peor_%_filas": pct_cat.iloc[0]["columna"] if len(pct_cat) else None,
        "%_combinaciones_cantidad_desfasada": pct_cant["%_combinaciones_desfasadas"],
    })

print("\n=== Resumen cruzado ===")
display(pd.DataFrame(impactos))


## 6. Valores solo en una tabla (detalle por columna)

Para cada variable inconsistente: lista de valores que están **solo en 2025** o **solo en 2026**, ordenados por cuántas filas cubren.


In [ ]:
TOP = 15
for tipo, pack in datos.items():
    print("\n" + "#" * 70)
    print(f"# {tipo.upper()} — tops de valores desfasados")
    print("#" * 70)
    tops = tops_desfasajes(pack["desfasajes_df"], top=TOP)
    pack["tops"] = tops
    for col, df_top in tops.items():
        n_vals = len(pack["desfasajes_df"].query("columna == @col"))
        n_filas = int(pack["desfasajes_df"].query("columna == @col")["n_filas"].sum())
        print(f"\n=== {col} — {n_vals} valores desfasados, {n_filas:,} filas ===")
        display(df_top)


## 7. Tabla consolidada de inconsistencias (ambas poblaciones)

Una sola vista: tipo × columna × % valores × % filas × cuántos solo_2025 / solo_2026.


In [ ]:
filas_consolidadas = []
for tipo, pack in datos.items():
    des_df = pack["desfasajes_df"]
    pct = pack["pct_categorias"]
    if des_df is None or des_df.empty:
        continue
    conteo = des_df.groupby(["columna", "solo_en"]).size().unstack(fill_value=0)
    for _, row in pct.iterrows():
        col = row["columna"]
        filas_consolidadas.append({
            "tipo": tipo,
            "columna": col,
            "valores_desfasados": int(row["valores_desfasados"]),
            "valores_unicos_totales": int(row["valores_unicos_totales"]),
            "%_valores": row["%_valores"],
            "filas_desfasadas": int(row["filas_desfasadas"]),
            "%_filas": row["%_filas"],
            "solo_2025": int(conteo.loc[col, "2025"]) if col in conteo.index and "2025" in conteo.columns else 0,
            "solo_2026": int(conteo.loc[col, "2026"]) if col in conteo.index and "2026" in conteo.columns else 0,
        })

tabla_inconsistencias = (
    pd.DataFrame(filas_consolidadas)
    .sort_values(["tipo", "%_filas"], ascending=[True, False])
    .reset_index(drop=True)
)
display(tabla_inconsistencias)

# Opcional: guardar el resumen junto a reportes/
out = CARPETA_DATOS / "reportes" / "inconsistencias_2025_vs_2026.csv"
out.parent.mkdir(parents=True, exist_ok=True)
tabla_inconsistencias.to_csv(out, index=False)
print("Guardado:", out)
